# Phase 0.4.26 — TS GFS Acquisition and Dataset Construction (Colab)

This notebook acquires the 300 NEW TS event groups from `docs/PHASE_0_4_24_TS_300_EVENT_MANIFEST.csv`, verifies and extracts them, and builds the final `phase_0_4_26_train.csv`/`phase_0_4_26_holdout.csv`, persisting everything to Google Drive in small batches. It does **not** train anything — training is a separate, later notebook.

## HOW TO RUN

1. Open this notebook in Google Colab.
2. Select the intended runtime: **Runtime → Change runtime type → CPU** (no GPU/A100 needed for this notebook).
3. Run cells from top to bottom, in order.
4. Authenticate Google Drive when prompted (a browser popup/consent flow).
5. Confirm the storage gate (Section 6) before proceeding — it checks real free disk space and will stop with a clear message if it isn't safe to continue.
6. Start Batch 1 (Section 9) and let it run.
7. If Colab disconnects, simply re-run the notebook from the top — Section 6 onward will detect completed work via the persistent state file and resume without re-downloading anything already finished.
8. **Do not manually delete files from the `/content/drive/MyDrive/SIH-Historical-GFS/` directory** while this notebook is running or between runs — the state file assumes nothing there has moved.
9. At completion (Section 19), save or download the final evidence report cell's output.

**This will take a long time.** GFS files are hundreds of MB each and there are ~600 of them across 30 batches — expect this to span multiple Colab sessions. That is expected and handled by the resumability design in Section 6/15.


## 1. Environment setup

In [ ]:
import sys, os, platform, json, shutil, hashlib, time
from pathlib import Path
from datetime import datetime, timezone

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Timestamp (UTC):", datetime.now(timezone.utc).isoformat())


## 2. Google Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/SIH-Historical-GFS')
print("Drive root will be:", DRIVE_ROOT)


## 3. Repository clone/setup

Replace `REPO_URL` below with the actual repository URL or a Drive-synced copy path if you prefer not to clone from a remote. This notebook does not assume which; it only needs `REPO_ROOT` to end up pointing at a working checkout that contains `scripts/`, `docs/`, and `SIH_PANINDIA_GRID_LABELS_20260930_143541Z/`.

In [ ]:
REPO_URL = "REPLACE_WITH_YOUR_REPO_URL_OR_LEAVE_BLANK_TO_USE_DRIVE_COPY"
REPO_DIR = Path('/content/SIH-Hyperlocal-Warning')

if REPO_URL and not REPO_DIR.exists():
    get_ipython().system(f'git clone {REPO_URL} {REPO_DIR}')
elif not REPO_DIR.exists():
    raise RuntimeError(
        "REPO_DIR does not exist and REPO_URL is blank. Either set REPO_URL above, "
        "or copy/symlink your repository checkout to /content/SIH-Hyperlocal-Warning "
        "before continuing. This notebook does not guess a location."
    )

REPO_ROOT = REPO_DIR
print("REPO_ROOT:", REPO_ROOT)
assert (REPO_ROOT / "scripts").exists(), "scripts/ not found under REPO_ROOT -- check your clone/copy"
assert (REPO_ROOT / "docs").exists(), "docs/ not found under REPO_ROOT -- check your clone/copy"


## 4. Dependency installation

In [ ]:
get_ipython().system('pip install -q eccodes pandas numpy')

import eccodes
import pandas as pd
import numpy as np
print("eccodes:", eccodes.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 4b. TS labels (Google Drive-hosted, never in Git)

`ts_labels.csv` is the real, already-existing TS label archive. It is deliberately NOT in this Git repository (per the Phase 0.2-0.4.26 preservation-staging decision, this file is held back from Git for now) and is NOT regenerated, copied, or substituted here -- it is read from Google Drive, where it was uploaded directly. This cell resolves that path, hard-fails if the file is missing, and verifies it is a readable CSV with the columns `build_phase_0_4_20_dataset.py`'s extraction contract actually requires (`cell_id`, `timestamp`, `label`, `label_status`, via `build_vobl_historical_gfs_ts_join.py::build_pilot()`). Every later call to `build_phase_0_4_20_dataset.py` in this notebook passes this exact path via `--ts-labels` -- the builder's own default fallback (`processed/labels/ts_labels.csv`, not present in this repo checkout) is never used.

In [ ]:
TS_LABELS_DRIVE_PATH = Path("/content/drive/MyDrive/ts_labels.csv")

if not TS_LABELS_DRIVE_PATH.exists():
    raise RuntimeError(
        f"ts_labels.csv not found at {TS_LABELS_DRIVE_PATH}. This file is Drive-hosted, "
        "not in Git -- upload it to the root of your Google Drive (My Drive) as "
        "'ts_labels.csv' before continuing. Refusing to fall back to any other path."
    )

_ts_labels_df = pd.read_csv(TS_LABELS_DRIVE_PATH)
_required_ts_label_columns = ["cell_id", "timestamp", "label", "label_status"]
_missing_ts_label_columns = [c for c in _required_ts_label_columns if c not in _ts_labels_df.columns]
if _missing_ts_label_columns:
    raise RuntimeError(
        f"{TS_LABELS_DRIVE_PATH} is missing required column(s) {_missing_ts_label_columns} "
        f"-- found columns: {list(_ts_labels_df.columns)}. This does not look like the real "
        "ts_labels.csv archive; refusing to proceed."
    )

print(f"ts_labels.csv found at {TS_LABELS_DRIVE_PATH}: {len(_ts_labels_df)} rows, "
      f"columns OK ({_required_ts_label_columns} all present).")
print("This path will be passed explicitly to every build_phase_0_4_20_dataset.py call "
      "via --ts-labels; its own default fallback path is never used.")


## 5. Environment verification

Confirms the existing, unmodified acquisition/extraction/validation scripts actually import on this runtime, exactly as verified on a Linux sandbox in Phase 0.4.25. This does not download anything -- it only checks that the code this notebook depends on is usable here.

In [ ]:
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import importlib
required_modules = [
    "acquire_phase_0_4_19_batch",
    "verify_phase_0_4_19_batch",
    "build_phase_0_4_20_dataset",
    "build_vobl_historical_gfs_ts_join",
    "historical_gfs_thermodynamics",
    "historical_gfs_precipitation",
    "historical_dataset_split",
    "acquire_historical_gfs_pilot",
]
for m in required_modules:
    importlib.import_module(m)
    print(m, "OK")

print("\nAll required modules imported successfully -- environment verified.")


## 6. Storage verification (safety gate)

Checks **actual** free disk space on this Colab runtime -- does not assume a fixed capacity, per Phase 0.4.25's design. Computes a safe batch size from the real measured GFS file-size evidence (mean ~404MB, max ~557MB per file) and will reduce the default batch size, or stop with a clear message, if the default does not safely fit.

In [ ]:
MEASURED_MAX_FILE_BYTES = 556_624_838   # largest of the 40 already-acquired files (Phase 0.4.19/0.4.22)
MEASURED_MEAN_FILE_BYTES = 403_658_081

DEFAULT_BATCH_EVENT_GROUPS = 10   # 20 files/batch
MIN_BATCH_EVENT_GROUPS = 1        # 2 files/batch, absolute floor
SAFETY_HEADROOM_BYTES = 2 * 1024**3  # keep 2GB free beyond the worst-case batch

total, used, free = shutil.disk_usage('/content')
print(f"/content free space: {free / 1e9:.2f} GB")

def worst_case_batch_bytes(n_event_groups):
    return n_event_groups * 2 * MEASURED_MAX_FILE_BYTES

batch_size = DEFAULT_BATCH_EVENT_GROUPS
while batch_size >= MIN_BATCH_EVENT_GROUPS:
    needed = worst_case_batch_bytes(batch_size) + SAFETY_HEADROOM_BYTES
    if needed <= free:
        break
    batch_size -= 1

if batch_size < MIN_BATCH_EVENT_GROUPS:
    raise RuntimeError(
        f"Not enough free disk space on this Colab runtime even for a single event group "
        f"(need at least {(worst_case_batch_bytes(1) + SAFETY_HEADROOM_BYTES)/1e9:.2f} GB, "
        f"have {free/1e9:.2f} GB free). Stopping -- do not proceed. "
        f"Try a fresh runtime or a Colab Pro instance with more disk."
    )

if batch_size < DEFAULT_BATCH_EVENT_GROUPS:
    print(f"WARNING: reduced batch size from {DEFAULT_BATCH_EVENT_GROUPS} to {batch_size} "
          f"event groups based on actual free space ({free/1e9:.2f} GB).")
else:
    print(f"Default batch size of {DEFAULT_BATCH_EVENT_GROUPS} event groups "
          f"(worst case {worst_case_batch_bytes(DEFAULT_BATCH_EVENT_GROUPS)/1e9:.2f} GB) fits safely.")

BATCH_SIZE_EVENT_GROUPS = batch_size
print("BATCH_SIZE_EVENT_GROUPS =", BATCH_SIZE_EVENT_GROUPS)

TMP_RAW_DIR = Path('/content/tmp_raw')
TMP_RAW_DIR.mkdir(parents=True, exist_ok=True)


## 7. Manifest verification

Loads `docs/PHASE_0_4_24_TS_300_EVENT_MANIFEST.csv` as the authoritative NEW acquisition manifest -- this notebook never modifies it. Confirms the expected structure before using it, and confirms the existing 20 event groups are NOT present in it (they must not be re-acquired as if new).

In [ ]:
MANIFEST_PATH = REPO_ROOT / "docs" / "PHASE_0_4_24_TS_300_EVENT_MANIFEST.csv"
assert MANIFEST_PATH.exists(), f"Manifest not found at {MANIFEST_PATH}"

with open(MANIFEST_PATH, "rb") as f:
    manifest_sha256 = hashlib.sha256(f.read()).hexdigest()
print("Manifest SHA256:", manifest_sha256)
print("(Record this value in your evidence report -- it lets you confirm later that the exact "
      "same manifest was used throughout, even across multiple Colab sessions.)")

manifest_df = pd.read_csv(MANIFEST_PATH)
n_rows = len(manifest_df)
event_groups = manifest_df["event_group_key"].unique()
n_event_groups = len(event_groups)
n_positive = manifest_df[manifest_df["label"] == "POSITIVE"]["event_group_key"].nunique()
n_negative = manifest_df[manifest_df["label"] == "NEGATIVE_CONFIRMED"]["event_group_key"].nunique()
n_train = manifest_df[manifest_df["partition"] == "train"]["event_group_key"].nunique()
n_holdout = manifest_df[manifest_df["partition"] == "holdout"]["event_group_key"].nunique()

print(f"Manifest rows (files): {n_rows}")
print(f"Manifest event groups: {n_event_groups}")
print(f"Positive event groups: {n_positive}")
print(f"Negative event groups: {n_negative}")
print(f"Train event groups: {n_train}")
print(f"Holdout event groups: {n_holdout}")

assert n_event_groups == 300, f"Expected 300 event groups, found {n_event_groups}"
assert n_positive == 150, f"Expected 150 positive, found {n_positive}"
assert n_negative == 150, f"Expected 150 negative, found {n_negative}"
assert n_train == 240, f"Expected 240 train, found {n_train}"
assert n_holdout == 60, f"Expected 60 holdout, found {n_holdout}"
assert 590 <= n_rows <= 610, f"Expected ~600 file rows, found {n_rows}"

EXISTING_20_EVENT_GROUPS = {
    "IND_13.0_78.0|2015-02-28|3", "IND_13.0_78.0|2016-03-13|2", "IND_13.0_78.0|2017-04-01|3",
    "IND_13.0_78.0|2017-06-05|2", "IND_13.0_78.0|2018-02-09|3", "IND_13.0_78.0|2018-07-17|2",
    "IND_13.0_78.0|2019-10-02|2", "IND_13.0_78.0|2020-03-20|2", "IND_13.0_78.0|2021-02-19|2",
    "IND_13.0_78.0|2021-02-20|2", "IND_13.0_78.0|2021-10-01|0", "IND_13.0_78.0|2022-03-20|2",
    "IND_13.0_78.0|2023-03-16|3", "IND_13.0_78.0|2023-11-01|1", "IND_13.0_78.0|2024-01-01|0",
    "IND_13.0_78.0|2024-01-01|1", "IND_13.0_78.0|2024-01-01|2", "IND_13.0_78.0|2024-05-12|2",
    "IND_13.0_78.0|2024-06-02|2", "IND_13.0_78.0|2024-06-03|2",
}
overlap = set(event_groups) & EXISTING_20_EVENT_GROUPS
assert len(overlap) == 0, f"Manifest must not include already-acquired event groups, found overlap: {overlap}"

print("\nManifest verification PASSED -- 300/150/150/240/60 confirmed, no overlap with existing 20.")


## 8. Batch configuration

Splits the 300 event groups into batches of `BATCH_SIZE_EVENT_GROUPS` (from Section 6), preserving the manifest's own partition/label assignment -- batching is purely for safe, incremental acquisition, it does not re-select or re-balance anything from Phase 0.4.24's design.

In [ ]:
event_group_order = list(dict.fromkeys(manifest_df.sort_values("selection_rank")["event_group_key"]))
batches = [event_group_order[i:i + BATCH_SIZE_EVENT_GROUPS] for i in range(0, len(event_group_order), BATCH_SIZE_EVENT_GROUPS)]
print(f"{len(batches)} batches of up to {BATCH_SIZE_EVENT_GROUPS} event groups each")
for i, b in enumerate(batches[:3]):
    print(f"  batch {i}: {len(b)} event groups")
print("  ...")


## 8b. URL construction preflight (Phase 0.4.26 bugfix)

Before any network download starts, every `expected_url` in the manifest is re-derived from `expected_filename` using the single canonical URL builder (`build_direct_url`, `scripts/acquire_historical_gfs_pilot.py` -- the same helper that successfully acquired the original 40 GFS files in Phase 0.4.19). If any row disagrees, the notebook stops here and downloads nothing. This cell does not change the manifest, event selection, labels, or partitioning -- it only validates the URL column that is actually used for acquisition.

In [ ]:
# Phase 0.4.26 bugfix preflight -- validates expected_url against the single
# canonical URL builder (build_direct_url, acquire_historical_gfs_pilot.py)
# BEFORE any network download starts. Read-only: does not change the manifest,
# selection, labels, or partitioning.
import re
from acquire_historical_gfs_pilot import build_direct_url

_FNAME_RE = re.compile(r"^gfs\.0p25\.(\d{10})\.f(\d{3})\.grib2$")

def validate_manifest_urls(df, n_samples=5):
    mismatches = []
    for _, row in df.iterrows():
        m = _FNAME_RE.match(row["expected_filename"])
        assert m, f"unexpected filename format: {row['expected_filename']!r}"
        cycle, lead = m.group(1), m.group(2)
        canonical = build_direct_url(cycle, lead)
        if canonical != row["expected_url"]:
            mismatches.append((row["expected_filename"], row["expected_url"], canonical))
    print(f"Checked {len(df)} manifest rows against the canonical URL builder.")
    print(f"Sample URLs (first {n_samples}):")
    for url in df["expected_url"].head(n_samples):
        print(" ", url)
    if mismatches:
        print(f"\nMISMATCH: {len(mismatches)} manifest row(s) disagree with the canonical builder.")
        for fname, manifest_url, canonical_url in mismatches[:10]:
            print(f"  {fname}:\n    manifest : {manifest_url}\n    canonical: {canonical_url}")
    return mismatches

_url_mismatches = validate_manifest_urls(manifest_df)
URLS_VALID = len(_url_mismatches) == 0
assert URLS_VALID, (
    f"{len(_url_mismatches)} manifest expected_url value(s) disagree with the canonical "
    "URL builder -- refusing to start batch acquisition. Fix the manifest/generator before "
    "re-running this notebook."
)
print("\nURL construction preflight PASSED -- safe to proceed to batch acquisition.")


## 9. Batch acquisition

Downloads one batch's GFS files to the **temporary** Colab disk (`/content/tmp_raw`), never to Drive and never all 600 at once. Uses the manifest's own `expected_url`/`expected_filename` -- no URL is invented here.

In [ ]:
import urllib.request

STATE_PATH = DRIVE_ROOT / "manifests" / "phase_0_4_26_batch_state.json"

def load_state():
    if STATE_PATH.exists():
        return json.loads(STATE_PATH.read_text())
    return {}

def save_state(state):
    STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True))

def download_file(url, dest_path):
    req = urllib.request.Request(url, headers={"User-Agent": "SIH-Hyperlocal-Warning/phase_0_4_26"})
    with urllib.request.urlopen(req, timeout=120) as resp, open(dest_path, "wb") as out:
        shutil.copyfileobj(resp, out)
    return dest_path.stat().st_size

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_batch(batch_event_groups, state):
    rows = manifest_df[manifest_df["event_group_key"].isin(batch_event_groups)]
    for _, row in rows.iterrows():
        fname = row["expected_filename"]
        key = fname
        if state.get(key, {}).get("status") in ("DOWNLOADED", "VERIFIED", "EXTRACTED", "VALIDATED", "PERSISTED", "PURGED"):
            continue  # resumability: already done in a previous session
        dest = TMP_RAW_DIR / fname
        print("Downloading", fname, "...")
        try:
            size = download_file(row["expected_url"], dest)
            state[key] = {
                "event_group_key": row["event_group_key"], "label": row["label"], "partition": row["partition"],
                "source_filename": fname, "source_url": row["expected_url"],
                "gfs_cycle": row["gfs_init_cycle_utc"], "forecast_lead": row["forecast_lead"],
                "file_size": size, "status": "DOWNLOADED", "batch_id": None,
                "downloaded_at": datetime.now(timezone.utc).isoformat(),
            }
        except Exception as e:
            state[key] = state.get(key, {})
            state[key].update({"status": "FAILED", "error": str(e), "event_group_key": row["event_group_key"]})
            print("FAILED:", fname, "--", e)
    save_state(state)
    return state

print("download_batch() defined -- run Section 16's loop to actually execute batches.")


## 10. GFS verification

Verifies each downloaded file's SHA256 and confirms it is a structurally valid GRIB2 file (readable by eccodes) before any extraction is attempted. Failed files are marked `FAILED` and left on disk for diagnosis -- never silently skipped or repaired.

In [ ]:
def verify_file(path):
    """Returns (ok: bool, reason: str)."""
    if not path.exists():
        return False, "file missing"
    try:
        with open(path, "rb") as f:
            gid = eccodes.codes_grib_new_from_file(f)
            if gid is None:
                return False, "no GRIB message found"
            eccodes.codes_release(gid)
    except Exception as e:
        return False, f"GRIB2 structural read failed: {e}"
    return True, "ok"

def verify_batch(batch_event_groups, state):
    rows = manifest_df[manifest_df["event_group_key"].isin(batch_event_groups)]
    for _, row in rows.iterrows():
        fname = row["expected_filename"]
        entry = state.get(fname)
        if entry is None or entry.get("status") != "DOWNLOADED":
            continue
        dest = TMP_RAW_DIR / fname
        ok, reason = verify_file(dest)
        if ok:
            entry["sha256"] = sha256_of(dest)
            entry["status"] = "VERIFIED"
            entry["verified_at"] = datetime.now(timezone.utc).isoformat()
        else:
            entry["status"] = "FAILED"
            entry["error"] = reason
            print("VERIFY FAILED:", fname, "--", reason, "(left on disk for diagnosis)")
    save_state(state)
    return state

print("verify_batch() defined.")


## 11. Historical feature extraction

Runs the existing, unmodified `build_vobl_historical_gfs_ts_join.py` extraction logic (via `build_phase_0_4_20_dataset.py`'s CLI, pointed at the Colab temp raw directory) against only the event groups in this batch that have fully VERIFIED files for both leads (f003+f006).

In [ ]:
def extraction_ready_event_groups(batch_event_groups, state):
    ready = []
    for eg in batch_event_groups:
        files = manifest_df[manifest_df["event_group_key"] == eg]["expected_filename"].tolist()
        if all(state.get(f, {}).get("status") == "VERIFIED" for f in files) and len(files) == 2:
            ready.append(eg)
    return ready

def extract_batch(batch_event_groups, batch_id, state):
    ready = extraction_ready_event_groups(batch_event_groups, state)
    if not ready:
        print("No event groups ready for extraction in this batch (nothing fully VERIFIED).")
        return state, None
    batch_out_dir = Path(f"/content/tmp_processed/batch_{batch_id:03d}")
    batch_out_dir.mkdir(parents=True, exist_ok=True)
    ready_mask = manifest_df["event_group_key"].isin(ready)
    batch_manifest_path = batch_out_dir / "batch_candidate_manifest.csv"
    out_df = manifest_df[ready_mask].copy()
    # Phase 0.4.26 bugfix: build_phase_0_4_20_dataset.py expects selected_gfs_cycle /
    # selected_lead (Phase 0.4.17 manifest schema); the Phase 0.4.24 manifest carries
    # the same information under gfs_init_cycle_utc / forecast_lead instead. Derive the
    # two aliases here -- the Phase 0.4.24 manifest itself is never modified.
    out_df["selected_gfs_cycle"] = pd.to_datetime(out_df["gfs_init_cycle_utc"]).dt.strftime("%Y%m%d%H")
    out_df["selected_lead"] = out_df["forecast_lead"]
    out_df.to_csv(batch_manifest_path, index=False)

    cmd = (
        f'python3 "{REPO_ROOT}/scripts/build_phase_0_4_20_dataset.py" '
        f'--partition all '
        f'--candidate-manifest "{batch_manifest_path}" '
        f'--raw-root "{TMP_RAW_DIR}" '
        f'--out-dir "{batch_out_dir}" '
        # Explicit --ts-labels always passed: the Drive-hosted file, never the
        # builder's own default (processed/labels/ts_labels.csv), which is not in
        # this repo checkout.
        f'--ts-labels "{TS_LABELS_DRIVE_PATH}"'
    )
    print("Running:", cmd)
    rc = os.system(cmd)
    for eg in ready:
        files = manifest_df[manifest_df["event_group_key"] == eg]["expected_filename"].tolist()
        for f in files:
            state[f]["status"] = "EXTRACTED" if rc == 0 else "FAILED"
            state[f]["extracted_at"] = datetime.now(timezone.utc).isoformat()
    save_state(state)
    return state, batch_out_dir

print("extract_batch() defined.")


## 12. Dataset validation

Runs the existing, unmodified leakage/partition checks (`historical_dataset_split.assign_partition` / `validate_no_event_group_key_crosses_partitions`) against the batch's extracted rows before anything is persisted. Hard-fails rather than silently repairing on any violation.

In [ ]:
from historical_dataset_split import assign_partition, validate_no_event_group_key_crosses_partitions

def validate_batch(batch_out_dir, state, batch_event_groups):
    if batch_out_dir is None:
        return state, False
    produced_csvs = list(batch_out_dir.glob("*.csv"))
    produced_csvs = [p for p in produced_csvs if p.name != "batch_candidate_manifest.csv"]
    if not produced_csvs:
        print("No dataset CSV produced for this batch -- marking extracted files FAILED.")
        for eg in batch_event_groups:
            for f in manifest_df[manifest_df["event_group_key"] == eg]["expected_filename"]:
                if state.get(f, {}).get("status") == "EXTRACTED":
                    state[f]["status"] = "FAILED"
                    state[f]["error"] = "no dataset CSV produced"
        save_state(state)
        return state, False

    all_ok = True
    for csv_path in produced_csvs:
        df = pd.read_csv(csv_path)
        if df.empty:
            continue
        partitions_claimed = set(df["event_group_key"].map(lambda k: assign_partition(k.split("|")[1])))
        # hard-fail check: no event_group_key spans more than one partition
        by_key = df.groupby("event_group_key")["ist_date"].apply(lambda s: {assign_partition(d) for d in s})
        bad = {k: v for k, v in by_key.items() if len(v) > 1}
        if bad:
            print("HARD FAIL: event_group_key(s) spanning multiple partitions:", bad)
            all_ok = False

    status = "VALIDATED" if all_ok else "FAILED"
    for eg in batch_event_groups:
        for f in manifest_df[manifest_df["event_group_key"] == eg]["expected_filename"]:
            if state.get(f, {}).get("status") == "EXTRACTED":
                state[f]["status"] = status
                state[f]["validated_at"] = datetime.now(timezone.utc).isoformat()
    save_state(state)
    return state, all_ok

print("validate_batch() defined.")


## 13. Persistent Drive write

Writes the batch's processed rows, provenance, and logs to Google Drive. Nothing is purged until Section 14's verification (re-reading the just-written Drive copy) confirms the write actually succeeded.

In [ ]:
def persist_batch(batch_out_dir, batch_id, state, batch_event_groups):
    if batch_out_dir is None:
        return state, False
    dest_dir = DRIVE_ROOT / "processed_batches" / f"batch_{batch_id:03d}"
    dest_dir.mkdir(parents=True, exist_ok=True)
    for p in batch_out_dir.glob("*"):
        shutil.copy2(p, dest_dir / p.name)

    provenance_dir = DRIVE_ROOT / "provenance"
    provenance_dir.mkdir(parents=True, exist_ok=True)
    batch_provenance = {f: state[f] for eg in batch_event_groups
                         for f in manifest_df[manifest_df["event_group_key"] == eg]["expected_filename"]
                         if f in state}
    (provenance_dir / f"batch_{batch_id:03d}_provenance.json").write_text(
        json.dumps(batch_provenance, indent=2, sort_keys=True)
    )

    logs_dir = DRIVE_ROOT / "logs"
    logs_dir.mkdir(parents=True, exist_ok=True)
    (logs_dir / f"batch_{batch_id:03d}_log.txt").write_text(
        f"Batch {batch_id} processed at {datetime.now(timezone.utc).isoformat()}\n"
        f"Event groups: {batch_event_groups}\n"
    )
    for eg in batch_event_groups:
        for f in manifest_df[manifest_df["event_group_key"] == eg]["expected_filename"]:
            if state.get(f, {}).get("status") == "VALIDATED":
                state[f]["status"] = "PERSISTED"
                state[f]["persisted_at"] = datetime.now(timezone.utc).isoformat()
                state[f]["batch_id"] = batch_id
    save_state(state)
    return state, dest_dir

print("persist_batch() defined.")


## 14. Raw temporary-file purge

Deletes a batch's temporary raw GRIB files **only** after re-reading the just-written Drive copy and confirming it matches what was produced (step 5 of the Phase 0.4.25 batch-complete contract) -- never before, and never on the basis of the write call alone.

In [ ]:
def verify_persisted_copy(dest_dir, batch_out_dir):
    if dest_dir is None or batch_out_dir is None:
        return False
    for p in batch_out_dir.glob("*"):
        drive_copy = dest_dir / p.name
        if not drive_copy.exists():
            print("Persistence verification FAILED: missing", drive_copy)
            return False
        if drive_copy.stat().st_size != p.stat().st_size:
            print("Persistence verification FAILED: size mismatch for", drive_copy.name)
            return False
    return True

def purge_batch_raw(batch_event_groups, state, dest_dir, batch_out_dir):
    if not verify_persisted_copy(dest_dir, batch_out_dir):
        print("NOT purging -- persistent copy verification failed. Raw files kept for diagnosis.")
        return state
    for eg in batch_event_groups:
        for f in manifest_df[manifest_df["event_group_key"] == eg]["expected_filename"]:
            if state.get(f, {}).get("status") != "PERSISTED":
                continue  # only purge files that completed the full lifecycle
            raw_path = TMP_RAW_DIR / f
            if raw_path.exists():
                raw_path.unlink()
            state[f]["status"] = "PURGED"
            state[f]["purged_at"] = datetime.now(timezone.utc).isoformat()
    save_state(state)
    print("Raw temporary files purged for this batch.")
    return state

print("purge_batch_raw() defined.")


## 15. Batch state update

In [ ]:
def run_one_batch(batch_event_groups, batch_id, state):
    print(f"\n=== Batch {batch_id}: {len(batch_event_groups)} event groups ===")
    state = download_batch(batch_event_groups, state)
    state = verify_batch(batch_event_groups, state)
    state, out_dir = extract_batch(batch_event_groups, batch_id, state)
    state, ok = validate_batch(out_dir, state, batch_event_groups)
    if not ok:
        print(f"Batch {batch_id} FAILED validation -- NOT persisting, NOT purging. Raw/extracted files kept for diagnosis.")
        return state
    state, dest_dir = persist_batch(out_dir, batch_id, state, batch_event_groups)
    state = purge_batch_raw(batch_event_groups, state, dest_dir, out_dir)
    print(f"Batch {batch_id} complete.")
    return state

print("run_one_batch() defined -- this is the full lifecycle for one batch, per Phase 0.4.25 Part 5/7.")


## 16. Repeat remaining batches

Runs every batch not yet fully PURGED. Safe to re-run this cell after a disconnect -- already-PURGED batches are skipped automatically via the state file.

In [ ]:
state = load_state()

def batch_is_complete(batch_event_groups, state):
    for eg in batch_event_groups:
        for f in manifest_df[manifest_df["event_group_key"] == eg]["expected_filename"]:
            if state.get(f, {}).get("status") != "PURGED":
                return False
    return True

for batch_id, batch_event_groups in enumerate(batches):
    if batch_is_complete(batch_event_groups, state):
        print(f"Batch {batch_id} already complete (PURGED) -- skipping.")
        continue
    state = run_one_batch(batch_event_groups, batch_id, state)
    # Re-check free disk space between batches, in case it has changed.
    _, _, free_now = shutil.disk_usage('/content')
    print(f"Free disk after batch {batch_id}: {free_now / 1e9:.2f} GB")

print("\nAll batches attempted. Check the state file for any FAILED entries before proceeding.")


## 17. Final train/holdout assembly

Combines the existing 20 already-acquired event groups (loaded from their existing `phase_0_4_20_train.csv`/`phase_0_4_20_holdout.csv`, if available in this checkout) with the newly acquired 300, **exactly once each** -- no event group is duplicated across the combination.

In [ ]:
processed_batches_dir = DRIVE_ROOT / "processed_batches"
all_new_rows = []
for batch_dir in sorted(processed_batches_dir.glob("batch_*")):
    for csv_path in batch_dir.glob("*.csv"):
        if csv_path.name == "batch_candidate_manifest.csv":
            continue
        df = pd.read_csv(csv_path)
        all_new_rows.append(df)

new_df = pd.concat(all_new_rows, ignore_index=True) if all_new_rows else pd.DataFrame()
print(f"New rows assembled from Drive-persisted batches: {len(new_df)}")

existing_train_path = REPO_ROOT / "data" / "external" / "historical_gfs" / "phase_0_4_20_train.csv"
existing_holdout_path = REPO_ROOT / "data" / "external" / "historical_gfs" / "phase_0_4_20_holdout.csv"
existing_frames = []
if existing_train_path.exists():
    existing_frames.append(pd.read_csv(existing_train_path))
if existing_holdout_path.exists():
    existing_frames.append(pd.read_csv(existing_holdout_path))
existing_df = pd.concat(existing_frames, ignore_index=True) if existing_frames else pd.DataFrame()
print(f"Existing (already-acquired 20-group) rows found: {len(existing_df)}")

combined = pd.concat([existing_df, new_df], ignore_index=True) if len(existing_df) or len(new_df) else pd.DataFrame()
dupes = combined[combined.duplicated(subset=["event_group_key"], keep=False)] if not combined.empty else combined
if not dupes.empty:
    print("WARNING: duplicate event_group_key values found across existing+new -- NOT deduplicating automatically:")
    print(dupes["event_group_key"].unique())
else:
    print("No duplicate event_group_key values between existing and new data.")

train_final = combined[combined["event_group_key"].map(lambda k: assign_partition(k.split("|")[1])) == "train"] if not combined.empty else combined
holdout_final = combined[combined["event_group_key"].map(lambda k: assign_partition(k.split("|")[1])) == "holdout"] if not combined.empty else combined

train_out = DRIVE_ROOT / "train" / "phase_0_4_26_train.csv"
holdout_out = DRIVE_ROOT / "holdout" / "phase_0_4_26_holdout.csv"
train_out.parent.mkdir(parents=True, exist_ok=True)
holdout_out.parent.mkdir(parents=True, exist_ok=True)
train_final.to_csv(train_out, index=False)
holdout_final.to_csv(holdout_out, index=False)
print(f"Wrote {train_out} ({len(train_final)} rows)")
print(f"Wrote {holdout_out} ({len(holdout_final)} rows)")


## 18. Final dataset validation

Hard-fails on any of: duplicate event_group, train/holdout overlap, label mismatch, missing required feature, unexpected source file, corrupted GRIB (already caught per-batch in Section 10), provenance mismatch, incomplete persistent write. Does **not** silently repair anything -- a failure here means stopping and investigating, not patching the data.

In [ ]:
FEATURE_COLUMNS = [
    "cape", "cin", "pwat_mm", "u850", "v850", "u200", "v200", "wind_shear_850_200_ms",
    "t850_k", "t700_k", "t500_k", "rh850_pct", "rh700_pct", "k_index", "totals_totals",
    "gfs_prate_kg_m2_s", "gfs_precip_3h_interval_mm",
]

failures = []

train_keys = set(train_final["event_group_key"]) if not train_final.empty else set()
holdout_keys = set(holdout_final["event_group_key"]) if not holdout_final.empty else set()
cross_overlap = train_keys & holdout_keys
if cross_overlap:
    failures.append(f"train/holdout overlap: {cross_overlap}")

for df_name, df in [("train", train_final), ("holdout", holdout_final)]:
    if df.empty:
        continue
    dup = df[df.duplicated(subset=["event_group_key"], keep=False)]
    if not dup.empty:
        failures.append(f"duplicate event_group_key in {df_name}: {dup['event_group_key'].unique().tolist()}")
    missing_features = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing_features:
        failures.append(f"missing required feature columns in {df_name}: {missing_features}")
    elif df[FEATURE_COLUMNS].isnull().any().any():
        incomplete_rows = df[df[FEATURE_COLUMNS].isnull().any(axis=1)]["event_group_key"].tolist()
        failures.append(f"incomplete feature rows in {df_name}: {incomplete_rows}")

failed_state_entries = {k: v for k, v in state.items() if v.get("status") == "FAILED"}
if failed_state_entries:
    failures.append(f"{len(failed_state_entries)} file(s) remain in FAILED state -- see provenance for diagnosis")

if failures:
    print("FINAL VALIDATION: FAILED")
    for f in failures:
        print(" -", f)
    raise RuntimeError("Final dataset validation failed -- see printed failures above. Not proceeding.")
else:
    print("FINAL VALIDATION: PASSED -- no duplicates, no train/holdout overlap, no missing features, no FAILED files.")


## 19. Final evidence report

Computes and prints the actual, measured numbers -- nothing here is assumed from the Phase 0.4.24 target counts.

In [ ]:
report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "manifest_sha256": manifest_sha256,
    "event_groups_total": len(train_keys | holdout_keys),
    "event_groups_train": len(train_keys),
    "event_groups_holdout": len(holdout_keys),
    "rows_train": len(train_final),
    "rows_holdout": len(holdout_final),
    "positive_rows_train": int((train_final["label"] == "POSITIVE").sum()) if "label" in train_final.columns and not train_final.empty else None,
    "negative_rows_train": int((train_final["label"] == "NEGATIVE_CONFIRMED").sum()) if "label" in train_final.columns and not train_final.empty else None,
    "positive_rows_holdout": int((holdout_final["label"] == "POSITIVE").sum()) if "label" in holdout_final.columns and not holdout_final.empty else None,
    "negative_rows_holdout": int((holdout_final["label"] == "NEGATIVE_CONFIRMED").sum()) if "label" in holdout_final.columns and not holdout_final.empty else None,
    "unknown_label_rows": int(((train_final.get("label", pd.Series(dtype=str)) == "UNKNOWN").sum()
                                 if not train_final.empty else 0)
                               + ((holdout_final.get("label", pd.Series(dtype=str)) == "UNKNOWN").sum()
                                  if not holdout_final.empty else 0)),
    "feature_completeness": "all FEATURE_COLUMNS present and non-null" if not failures else "FAILED (see Section 18)",
    "train_holdout_separation": "no overlap" if not cross_overlap else f"OVERLAP: {cross_overlap}",
    "duplicate_event_groups_found": bool(dupes.shape[0]) if not combined.empty else False,
    "failed_files_remaining": len(failed_state_entries),
    "final_validation": "PASSED" if not failures else "FAILED",
}

print(json.dumps(report, indent=2, sort_keys=True))

report_path = DRIVE_ROOT / "logs" / "phase_0_4_26_final_evidence_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2, sort_keys=True))
print(f"\nSaved to {report_path} -- download or copy this file to send back for audit.")
